# Phase 2 -- Data Quality Review

**WID Datathon 2026 -- Food System Resilience Agent**

Reviews the output of the Phase 2 Bronze -> Silver ELT pipeline
(`src/elt/pipeline.py`). Does **not** call the FAOSTAT API or transform
anything -- it loads the already-built `data/food_system.db`, the generated
`data_quality/*.csv`, and `reports/phase2_data_quality.md`.

Rebuild the inputs with:

```bash
uv run python -m src.elt.pipeline
```

No feature engineering, scoring, or Phase 3+ analysis happens here --
see `PHASE2.md` / `CLAUDE.md` for scope.


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from sqlalchemy import inspect, text

from src.database.connection import get_engine

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 120)

DQ = REPO_ROOT / "data_quality"
engine = get_engine(REPO_ROOT / "data" / "food_system.db")
insp = inspect(engine)
print("database:", engine.url)
print("tables:", len(insp.get_table_names()))


database: sqlite:////Users/shivanimac/Python/food-system-resilience-agent/data/food_system.db
tables: 22


## 1. Ingestion runs (`etl_runs`) -- one auditable row per domain

In [2]:
runs = pd.read_sql_query("SELECT * FROM etl_runs ORDER BY domain", engine)
runs[["domain", "priority", "status", "requested_start_year", "requested_end_year",
      "actual_min_year", "actual_max_year", "missing_requested_years",
      "requested_item_count", "retrieved_item_count", "country_count", "partner_count",
      "bronze_rows", "silver_rows", "error_message"]]


,domain,priority,status,requested_start_year,requested_end_year,actual_min_year,actual_max_year,missing_requested_years,requested_item_count,retrieved_item_count,country_count,partner_count,bronze_rows,silver_rows,error_message
0,CAHD,core,success,2014,2024,2017,2024,"[2014, 2015, 2016]",8.0,8,252,NaN,5702,5004,None
1,CP,supporting,success,2014,2024,2014,2024,[],2.0,2,252,NaN,53228,52856,None
2,FBS,core,success,2014,2024,2014,2023,[2024],NaN,122,213,NaN,2424763,1953306,None
3,FS,core,success,2014,2024,2014,2024,[],13.0,13,246,NaN,45203,22924,None
4,GT,supporting,success,2014,2024,2014,2023,[2024],45.0,44,281,NaN,582911,487913,None
5,QCL,core,success,2014,2024,2014,2024,[],162.0,161,244,NaN,432189,309946,None
6,RFM,supporting,success,2014,2024,2014,2024,[],25.0,25,248,256.0,1721174,1676964,None
7,RL,supporting,success,2014,2024,2014,2024,[],45.0,45,284,NaN,103076,88595,None
8,TM,core,success,2014,2024,2014,2024,[],154.0,154,198,221.0,6123614,5931780,None


`status` is per-domain and independent. A **core** domain
(QCL, TM, FBS, FS, CAHD) at `failed` blocks Phase 2 (non-zero pipeline exit
code); a **supporting** domain (CP, GT, RL, RFM) at `failed`/`partial` is
reported but does not.

## 2. QCL crops-only scope

In [3]:
from src.elt.qcl_items import DEFAULT_QCL_CROP_ITEMS_PATH, load_qcl_crop_items

crop_items = load_qcl_crop_items(DEFAULT_QCL_CROP_ITEMS_PATH)
print(f"{len(crop_items)} crop items (FAOSTAT itemgroup QC, leaf items only)")
pd.DataFrame(crop_items).head(10)


162 crop items (FAOSTAT itemgroup QC, leaf items only)


,item_code,item
0,101,Canary seed
1,103,Mixed grain
2,108,Cereals n.e.c.
3,116,Potatoes
4,122,Sweet potatoes
5,125,"Cassava, fresh"
6,135,Yautia
7,136,Taro
8,137,Yams
9,149,"Edible roots and tubers with high starch or inulin content, n.e.c...."


## 3. FAOSTAT flags -- preserved through Bronze, never used to drop rows

In [4]:
flags = pd.read_csv(DQ / "bronze_flag_summary.csv")
flags.pivot_table(index="domain", columns="flag", values="n_rows", aggfunc="sum", fill_value=0)


flag,A,E,I,L,M,O,Q,X
domain,,,,,,,,
CAHD,0,5063,0,3,0,492,144,0
CP,1653,26596,1181,24,0,0,0,23774
FBS,0,1489050,935713,0,0,0,0,0
FS,1060,33688,0,0,0,6944,3511,0
GT,9060,573851,0,0,0,0,0,0
QCL,239244,126063,53106,0,6790,0,0,6986
RFM,815443,287491,154936,5542,0,0,0,457762
RL,18741,45295,35720,79,0,0,0,3241
TM,6070921,16,49953,0,0,0,0,2724


## 4. Unit inventory and conversions

In [5]:
print("Bronze unit inventory (domain x unit x element):")
display(pd.read_csv(DQ / "bronze_unit_inventory.csv").head(40))
conv_path = DQ / "unit_conversions_applied.csv"
conv = pd.read_csv(conv_path)
if not conv.empty:
    print("\nUnit conversions applied in Silver:")
    display(conv)
else:
    print("\nNo unit conversions were applicable to the rows actually ingested.")


Bronze unit inventory (domain x unit x element):


,domain,unit,element,n_rows
0,QCL,t,Production,148146
1,QCL,ha,Area harvested,145906
2,QCL,kg/ha,Yield,138137
3,TM,1000 USD,Import value,1604261
4,TM,t,Import quantity,1604260
5,TM,1000 USD,Export value,1457547
6,TM,t,Export quantity,1457546
7,FBS,1000 t,Domestic supply quantity,237186
8,FBS,1000 t,Import quantity,228654
9,FBS,g/cap/d,Fat supply quantity (g/capita/day),224096



Unit conversions applied in Silver:


,domain,from_unit,to_unit,factor,elements,n_rows
0,FBS,1000 t,t,1000.0,"Production, Import quantity, Export quantity, Stock Variation, Dom...",1529904


## 5. Duplicates

In [6]:
dc = pd.read_csv(DQ / "duplicate_conflicts.csv")
cov = pd.read_csv(DQ / "duplicate_conflicts_coverage.csv")
print("Domains whose Silver transform ran in the invocation that wrote these files:")
display(cov)
if not dc.empty:
    print(f"{len(dc)} conflicting-duplicate key(s) resolved (prefer official flag, then largest |value|):")
    display(dc.head(30))
else:
    print("No conflicting duplicates found. Exact duplicates (if any) were collapsed silently in Silver.")


Domains whose Silver transform ran in the invocation that wrote these files:


,domain,n_conflicts


No conflicting duplicates found. Exact duplicates (if any) were collapsed silently in Silver.


## 6. Commodity mapping status

In [7]:
cm = pd.read_csv(DQ / "commodity_mapping.csv")
print(cm["mapping_status"].value_counts())
print("\nA sample of unmapped commodity items (kept visible, never force-mapped):")
cm[cm.mapping_status == "unmapped"][["domain_code", "item_code", "item"]].head(30)


mapping_status
unmapped          434
not_applicable    112
manual             28
Name: count, dtype: int64

A sample of unmapped commodity items (kept visible, never force-mapped):


,domain_code,item_code,item
0,QCL,101,Canary seed
1,QCL,103,Mixed grain
2,QCL,108,Cereals n.e.c.
6,QCL,135,Yautia
7,QCL,136,Taro
8,QCL,137,Yams
9,QCL,149,"Edible roots and tubers with high starch or inulin content, n.e.c...."
13,QCL,161,Other sugar crops n.e.c.
14,QCL,176,"Beans, dry"
15,QCL,181,"Broad beans and horse beans, dry"


## 7. Historical coverage (country level) -- reported, not filtered

In [8]:
hc_path = DQ / "historical_coverage_country.csv"
hc = pd.read_csv(hc_path)
if not hc.empty:
    n_insufficient = int((~hc.sufficient_history).sum())
    print(f"{len(hc)} countries with QCL production data; {n_insufficient} flagged insufficient_history")
    display(hc.sort_values("coverage_pct").head(25))
else:
    print("silver_production is empty (QCL did not load) -- see etl_runs above.")


197 countries with QCL production data; 0 flagged insufficient_history


,area_code,first_year,last_year,n_years,expected_years,coverage_pct,sufficient_history
141,299,2014,2022,9,11,81.8,True
55,157,2014,2023,10,11,90.9,True
0,1,2014,2024,11,11,100.0,True
126,244,2014,2024,11,11,100.0,True
127,249,2014,2024,11,11,100.0,True
128,25,2014,2024,11,11,100.0,True
129,250,2014,2024,11,11,100.0,True
130,251,2014,2024,11,11,100.0,True
131,255,2014,2024,11,11,100.0,True
132,256,2014,2024,11,11,100.0,True


## 8. Missingness

In [9]:
pd.read_csv(DQ / "bronze_missingness.csv")

,domain,column,n_missing,n_total
0,QCL,reporter_area_code,432189,432189
1,QCL,reporter_area,432189,432189
2,QCL,partner_area_code,432189,432189
3,QCL,partner_area,432189,432189
4,QCL,month_code,432189,432189
5,QCL,month,432189,432189
6,QCL,source_code,432189,432189
7,QCL,source,432189,432189
8,QCL,release_code,432189,432189
9,QCL,release,432189,432189


## 9. Silver validation (domain-aware)

Hard checks (e.g. no livestock item may reach `silver_production`) block
core-table validity; soft checks are reported only -- see
`src/validation/silver_schemas.py`.

In [10]:
val = pd.read_csv(DQ / "silver_validation.csv")
hard = val[(val.severity == "hard") & (val.n_violations > 0)]
soft = val[(val.severity == "soft") & (val.n_violations > 0)]
print("HARD violations:", len(hard), "| SOFT (reported only):", len(soft))
display(val[val.n_violations > 0] if (val.n_violations > 0).any() else val.head(20))


HARD violations: 0 | SOFT (reported only): 0


,table,check,severity,n_violations,detail
0,silver_production,non_null:area_code,hard,0,NaN
1,silver_production,non_null:item_code,hard,0,NaN
2,silver_production,non_null:element,hard,0,NaN
3,silver_production,non_null:year,hard,0,NaN
4,silver_production,year_in_requested_range,soft,0,NaN
5,silver_production,area_code_resolves_to_silver_country,soft,0,NaN
6,silver_production,non_negative_where_required,soft,0,NaN
7,silver_production,qcl_crops_only_no_livestock_leak,hard,0,silver_production must contain only the 162-code crop-only item set
8,silver_trade,non_null:reporter_area_code,hard,0,NaN
9,silver_trade,non_null:partner_area_code,hard,0,NaN


## 10. Structural tables/views -- the Phase 3 inputs

In [11]:
for view in ["silver_country_year", "silver_country_commodity_year", "silver_trade_matrix"]:
    try:
        df = pd.read_sql_query(f"SELECT * FROM {view} LIMIT 5", engine)
        n = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {view}", engine)["n"].iloc[0]
        print(f"=== {view} ({n:,} rows) ===")
        display(df)
    except Exception as exc:  # noqa: BLE001
        print(f"{view}: not available ({exc})")


=== silver_country_year (2,273 rows) ===


,area_code,year
0,1,2014
1,1,2015
2,1,2016
3,1,2017
4,1,2018


=== silver_country_commodity_year (2,263,252 rows) ===


,area_code,item_code,year,source_domain
0,1,108,2014,QCL
1,1,108,2014,QCL
2,1,108,2014,QCL
3,1,108,2015,QCL
4,1,108,2015,QCL


=== silver_trade_matrix (5,931,780 rows) ===


,reporter_area_code,partner_area_code,item_code,element,year,value
0,1,185,101,Import quantity,2018,16.14
1,1,185,101,Import quantity,2024,9.70
2,1,185,101,Import value,2018,4.00
3,1,185,101,Import value,2024,7.00
4,1,230,101,Import quantity,2015,10.00


## 11. Requested vs retrieved analytical variables

`config/variables.yaml` (translated from `DATA.md`) is the source of truth for
what this project asked FAOSTAT for. Anything requested that did not come back
is listed here rather than quietly replaced with a substitute.

In [12]:
scope = pd.read_csv(DQ / "variable_scope.csv")
summary = scope.groupby(["domain", "scope_kind"]).agg(
    requested=("requested", "size"), retrieved=("retrieved", "sum")
).reset_index()
summary["unavailable"] = summary["requested"] - summary["retrieved"]
display(summary)
missing = scope[~scope["retrieved"]]
if missing.empty:
    print("Every requested variable was returned by the API.")
else:
    print(f"{len(missing)} requested variable(s) not returned by the API:")
    display(missing)


,domain,scope_kind,requested,retrieved,unavailable
0,CAHD,item,8,8,0
1,CP,item,2,2,0
2,FBS,element,14,14,0
3,FS,item,13,13,0
4,GT,item,45,44,1
5,QCL,element,3,3,0
6,QCL,item,162,161,1
7,RFM,element,4,4,0
8,RFM,item,25,25,0
9,RL,item,45,45,0


10 requested variable(s) not returned by the API:


,domain,scope_kind,requested,retrieved
160,QCL,item,839,False
207,TM,item,254,False
215,TM,item,277,False
221,TM,item,305,False
222,TM,item,310,False
223,TM,item,328,False
232,TM,item,378,False
271,TM,item,542,False
319,TM,item,800,False
433,GT,item,6966,False


## 12. Requested items that did not arrive -- and the proof of why

Where a domain retrieved fewer items than it requested, the pipeline probes
each missing item live rather than assuming. Only a shortfall *established* as
source behaviour is forgiven; anything unproven keeps the domain at `partial`.
See `PHASE2.md` section 15.

In [13]:
gaps = pd.read_csv(DQ / "scope_gaps.csv")
if gaps.empty:
    print("Every requested item arrived.")
else:
    print(gaps.groupby("classification").size().to_string())
    print()
    unexplained = gaps[gaps.counts_against_status]
    print(f"{len(unexplained)} gap(s) still counting against a domain's status"
          f"{' -- investigate these' if len(unexplained) else ' (all explained)'}")
    display(gaps)


classification
absent_at_source         1
absent_from_dimension    8
out_of_window            1

0 gap(s) still counting against a domain's status (all explained)


,domain,item_code,classification,counts_against_status,detail
0,GT,6966,absent_at_source,False,no observation in any year or country; the item filter is sound at...
1,QCL,839,out_of_window,False,published 1961-1990 (272 rows); nothing in the requested 2014-2024
2,TM,254,absent_from_dimension,False,not listed in TM's own item dimension
3,TM,277,absent_from_dimension,False,not listed in TM's own item dimension
4,TM,305,absent_from_dimension,False,not listed in TM's own item dimension
5,TM,310,absent_from_dimension,False,not listed in TM's own item dimension
6,TM,328,absent_from_dimension,False,not listed in TM's own item dimension
7,TM,378,absent_from_dimension,False,not listed in TM's own item dimension
8,TM,542,absent_from_dimension,False,not listed in TM's own item dimension
9,TM,800,absent_from_dimension,False,not listed in TM's own item dimension


## 13. Full per-domain report

In [14]:
print((REPO_ROOT / "reports" / "phase2_data_quality.md").read_text())

# Phase 2 Data Quality Report

Pipeline run: `867aa1ba-edbb-4c01-b56a-fd6bfd054ce6`

Requested analytical period: **2014-2024** for every domain. Years a domain does not publish are reported as missing, never interpolated or treated as an error.

## Per-domain summary

| Domain | Priority | Status | Requested years | Actual years | Missing years | Requested items | Retrieved items | Requested elements | Retrieved elements | Countries | Partners | Bronze rows | Silver rows |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| CAHD | core | success | 2014-2024 | 2017-2024 | 2014, 2015, 2016 | 8 | 8 | - | - | 252 | - | 5702 | 5004 |
| CP | supporting | success | 2014-2024 | 2014-2024 | - | 2 | 2 | - | - | 252 | - | 53228 | 52856 |
| FBS | core | success | 2014-2024 | 2014-2023 | 2024 | - | 122 | 14 | 14 | 213 | - | 2424763 | 1953306 |
| FS | core | success | 2014-2024 | 2014-2024 | - | 13 | 13 | - | - | 246 | - | 45203 | 22924 |
| GT | supporting | success | 2014-2024 | 2014-2023 